In [1]:
%pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.4 MB/s eta 0:00:0000:01


In [2]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import (GATConv, global_mean_pool)


class GraphEncoder(torch.nn.Module):

    def __init__(self, in_channels=4, hidden_dim=128, num_layers=3, heads=4):
        super().__init__()

        self.convs = torch.nn.ModuleList()

        self.convs.append(
            GATConv(
                in_channels=in_channels,
                out_channels=hidden_dim,
                heads=heads,
                dropout=0.2
            )
        )

        for _ in range(num_layers - 1):

            self.convs.append(
                GATConv(
                    in_channels=hidden_dim * heads,
                    out_channels=hidden_dim,
                    heads=heads,
                    dropout=0.2
                )
            )
    
    def forward(self, x, edge_index, batch):
        
        for conv in self.convs:

            x = conv(x, edge_index)

            x = F.relu(x)
        
        graph_embedding = global_mean_pool(x, batch)

        return graph_embedding

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class ValueHead(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim, 
                hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                1
            )
        )
    
    def forward(self, x):

        return self.mlp(x)

In [ ]:
import torch.nn as nn


class ChessGNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.encoder = GraphEncoder(
            in_channels=4,
            hidden_dim=128,
            num_layers=3,
            heads=4
        )

        self.value_head = ValueHead(input_dim=128 * 4)
    
    def forward(self, data):

        embedding = self.encoder(
            data.x,
            data.edge_index,
            data.batch
        )

        value = self.value_head(embedding)

        return value

In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
from pathlib import Path


PROJECT_ROOT = Path(
    "/content/drive/MyDrive/chess-ai"
)

processed = PROJECT_ROOT / "data" / "processed"

print(processed.exists())

if processed.exists():
    print(list(processed.iterdir()))

True
[PosixPath('/content/drive/MyDrive/chess-ai/data/processed/graph_dataset.pt')]


In [9]:
import torch
from torch_geometric.loader import DataLoader
from tqdm import tqdm
from pathlib import Path


PROJECT_ROOT = Path('/content/drive/MyDrive/chess-ai')
DATA_DIR = PROJECT_ROOT / 'data'
PROCESSED_DIR = DATA_DIR / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'

GRAPH_PATH = PROCESSED_DIR / 'graph_dataset.pt'
MODEL_PATH = PROJECT_ROOT / 'experiments' / 'value_model.pt'

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

# =========================
# SPLITTING & LOADING
# =========================
graphs = torch.load(GRAPH_PATH, weights_only=False)
OUTPUT_PATH = MODEL_PATH

n = len(graphs)

train_end = int(0.8 * n)
val_end = int(0.9 * n)

train_graphs = graphs[:train_end]
val_graphs = graphs[train_end:val_end]
test_graphs = graphs[val_end:]

#print(type(graphs[0]).__name__)

train_loader = DataLoader(train_graphs,
                          batch_size=64,
                          shuffle=True)

val_loader = DataLoader(val_graphs,
                        batch_size=64,
                        shuffle=False)

# ===============
# CUDA
# ===============
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#print(device)

# ==================
# HYPERPARAMETERS
# ==================
model = ChessGNN().to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

criterion = torch.nn.MSELoss()

EPOCHS = 30

# ================
# TRAINING LOOP
# ================
train_losses = []
val_losses = []

batch = next(iter(train_loader))
batch = batch.to(device)

with torch.no_grad():
    pred = model(batch)

print(pred[:15])

for epoch in range(EPOCHS):

    model.train()

    total_train_loss = 0

    for batch in tqdm(train_loader):
        
        batch = batch.to(device)

        optimizer.zero_grad()

        pred = model(batch)

        target = batch.y_value

        loss = criterion(pred.squeeze(), target.squeeze())

        loss.backward()

        optimizer.step()

        total_train_loss += loss.item()

    print(f'\nPREDICTIONS: {pred[:5]}\n')

    print(f'\nGRADIENTS:')
    for name, p in model.named_parameters():
        if p.grad is not None:
            print(name, p.grad.norm().item())
            break
    print('')

    avg_train_loss = total_train_loss / len(train_loader)

    train_losses.append(avg_train_loss)

    # Validation
    model.eval()

    total_val_loss = 0

    with torch.no_grad():

        for batch in val_loader:

            batch = batch.to(device)

            loss = criterion(model(batch).squeeze(), batch.y_value.squeeze())

            total_val_loss += loss
    
        avg_val_loss = total_val_loss / len(val_loader)

        val_losses.append(avg_val_loss)
    
    print(f'Epoch {epoch+1}: train loss = {avg_train_loss:.4f} | val loss = {avg_val_loss:.4f}')

# =================
# SAVING
# =================
torch.save(model.state_dict(), OUTPUT_PATH)

# ====================
# VALIDATION & TEST
# ====================
model.eval()

val_loss = 0

with torch.no_grad():

    for batch in val_loader:

        batch = batch.to(device)

        pred = model(batch)

        loss = criterion(pred.squeeze(), batch.y_value.squeeze())

        val_loss += loss.item()

print(f'Validation Loss: {val_loss / len(val_loader)}')

KeyboardInterrupt: 